# Intermediate 01: Identity Propagation and Delegated Authority

In simple systems, knowing *who is calling* is easy. In agentic architectures, a request might hop from the User -> Agent -> Document Service -> Storage Service.

If the Agent uses its own powerful infrastructure privileges to fetch data, it can be tricked into retrieving documents the original User isn't allowed to see. This is the **Confused Deputy** problem.

The solution is **Identity Propagation**: securely passing the user's identity and a tightly scoped **Delegation Grant** all the way down the chain. At each hop, the token is **exchanged** for a narrower downstream token, ensuring monotonic attenuation.

In [ ]:
import sys
from pathlib import Path
import importlib
sys.path.append(str(Path.cwd().parent.parent.parent / 'curriculum' / 'intermediate' / '01-identity-propagation'))
lab = importlib.import_module('01_identity_propagation')
from datetime import datetime, timezone, timedelta

class MutableClock:
    def __init__(self, start):
        self.now = start
    def __call__(self):
        return self.now
    def advance(self, minutes):
        self.now += timedelta(minutes=minutes)

clock = MutableClock(datetime(2025, 1, 1, 12, 0, tzinfo=timezone.utc))
ds = lab.DelegationService(clock_fn=clock)
audit = lab.AuditSink()
auth_alice = lab.ApplicationIdentityProvider.for_alice()
auth_bob = lab.ApplicationIdentityProvider.for_bob()
auth_mallory = lab.ApplicationIdentityProvider.for_mallory()
auth_agent = lab.InfrastructureIdentityProvider.for_research_agent()
auth_doc = lab.InfrastructureIdentityProvider.for_document_service()
auth_storage = lab.InfrastructureIdentityProvider.for_storage_service()
auth_evil = lab.InfrastructureIdentityProvider.for_evil_agent()
auth_job = lab.ServiceJobIdentityProvider.for_document_maintenance()

storage = lab.StorageService(ds, audit, auth_storage)
secure_docs = lab.SecureDocumentService(ds, storage, audit, auth_doc)
naive_docs = lab.NaiveDocumentService(storage, auth_doc)
app = lab.ResearchApplication(ds, secure_docs, auth_agent, naive_docs, audit)


## 1. Scenario and Identity Map

In this lab, we have three users:
- **Alice** (Tenant: Acme, Ops: read/comment, Docs: doc-101, doc-102)
- **Bob** (Tenant: Acme, Ops: read, Docs: doc-103)
- **Mallory** (Tenant: Globex, Ops: read, Docs: doc-globex-01)

And a highly classified document, `doc-secret`, that no user is allowed to access.

In [ ]:
print('--- Principals ---')
for pid, p in lab.PRINCIPAL_REGISTRY.items():
    print(f"{pid}: tenant={p.tenant}, ops={list(p.allowed_operations)}, resources={list(p.allowed_resources)}")
print('\n--- Workloads ---')
for wid, w in lab.WORKLOAD_REGISTRY.items():
    print(f"{wid}: tenant={w.tenant}")

## 2. Principal vs Workload Identity
A *Principal* is the human user (Alice). A *Workload Identity* is the software service (research-agent).
They are separate. One cannot replace the other.

## 3. Authentication vs Delegation vs Authorization
- **Authentication:** Proving you are who you say you are.
- **Delegation:** Granting permission to a workload to act on your behalf.
- **Authorization:** Deciding if a specific action is allowed based on the grant.

## 4. Naive Ambient Authority
First, let's see how a vulnerable service behaves. The `NaiveDocumentService` trusts the Research Agent's workload identity completely. It uses its *ambient* service-level authority to query storage, ignoring the user.

In [ ]:
my_auth = lab.InfrastructureIdentityProvider.for_document_service()
res = storage.read_object_naive(my_auth, 'doc-secret', 'naive-req')
print(f"Storage Naive Read Result: {res}")

## 5. Confused Deputy Exploit
Because Alice can talk to the Research Agent, and the agent can talk to the Naive Document Service using ambient authority, Alice can trick the agent into retrieving `doc-secret`!

In [ ]:
auth_alice = lab.ApplicationIdentityProvider.for_alice()
ans = app.answer_naive(auth_alice, 'doc-secret')
print(f"Exploit Result: {ans}")

## 6. Authoritative Principal Resolution
A secure delegation issuer accepts provider-verified principal and workload evidence, derives their identifiers, and then resolves authorization attributes from a trusted registry (like Entra ID or Okta). A registry lookup alone does not authenticate the caller.

In [ ]:
principal_id = 'alice'
print(f"Resolving {principal_id} -> {lab.PRINCIPAL_REGISTRY[principal_id]}")

## 7. Authoritative Workload Authentication
Similarly, a string `caller_id = 'document-service'` isn't proof of identity. Authentication requires trusted infrastructure (mTLS, SPIFFE). Our `InfrastructureIdentityProvider` represents this.

In [ ]:
auth_workload = lab.InfrastructureIdentityProvider.for_document_service()
print(f"Authenticated Context: {auth_workload}")

## 8. Authentication Context Substitution
An attacker might transplant a valid context ID onto another identity or recreate all visible fields. The simulation accepts only the canonical object injected by trusted middleware, so neither attack recreates authentication evidence.

In [ ]:
forged_alice = lab.AuthenticatedPrincipal('alice', auth_bob.auth_context_id)
print(f"Forged Principal substitution verified: {lab.ApplicationIdentityProvider.verify(forged_alice)}")

copied_bob = lab.AuthenticatedPrincipal('bob', 'ctx-bob')
print(f"Exact-copy Principal verified: {lab.ApplicationIdentityProvider.verify(copied_bob)}")

forged_doc = lab.AuthenticatedWorkload('document-service', 'acme', auth_agent.auth_context_id)
copied_doc = lab.AuthenticatedWorkload('document-service', 'acme', 'ctx-doc')
print(f"Forged Workload substitution verified: {lab.InfrastructureIdentityProvider.verify(forged_doc)}")
print(f"Exact-copy Workload verified: {lab.InfrastructureIdentityProvider.verify(copied_doc)}")

## 10. Delegation Issuance
When Alice makes a request, the app issues a tightly scoped **Delegation Grant** binding Alice to the Research Agent.

In [ ]:
grant_101 = ds.issue(
    principal_context=auth_alice,
    delegate_context=auth_agent,
    audience='document-service', 
    requested_operations={'read'}, 
    requested_resources={'doc-101'}
)
print(f"Issued Grant ID: {grant_101.grant.grant_id}")

## 10. Forged Principal Attack
If an attacker constructs an identity object with copied fields, the issuer rejects it before resolving authorization attributes.

In [ ]:
forged_principal = lab.AuthenticatedPrincipal('alice', 'ctx-alice')
fake_grant = ds.issue(forged_principal, auth_agent, 'document-service', {'read'}, {'doc-101'})
print(f"Forged Principal Grant: {fake_grant}")

## 11. Forged Grant Attack
If an attacker tries to construct a fake `DelegationGrant` object manually and pass it to a service, the service verifies it against the issuer's store. It will fail.

In [ ]:
fake_obj = lab.DelegationGrant('fake-123', None, 'alice', 'document-service', 'acme', 'storage-service', frozenset({'read'}), frozenset({'doc-secret'}), clock(), clock() + timedelta(minutes=60), 'hacker')
decision = ds.verify(fake_obj, 'document-service', 'storage-service', 'acme', 'read', 'doc-secret')
print(f"Forged Grant Verification: {decision.reason}")

## 12. Audience Restriction
A grant issued for `document-service` cannot be used directly against `storage-service`.

In [ ]:
decision = ds.verify(grant_101.grant, 'research-agent', 'storage-service', 'acme', 'read', 'doc-101')
print(f"Audience Mismatch: {decision.reason}")

## 13. Operation Down-Scoping
During issuance, you cannot request operations the principal doesn't have.

In [ ]:
grant = ds.issue(auth_alice, auth_agent, 'document-service', {'delete'}, {'doc-101'})
print(f"Escalated Operation Grant: {grant}")

## 14. Resource Down-Scoping
Similarly, you cannot request resources the principal doesn't possess.

In [ ]:
grant = ds.issue(auth_alice, auth_agent, 'document-service', {'read'}, {'doc-secret'})
print(f"Escalated Resource Grant: {grant}")

## 15. Tenant Binding
A user from Acme cannot delegate a workload from Globex, nor access a Globex resource.

In [ ]:
grant = ds.issue(auth_alice, auth_evil, 'document-service', {'read'}, {'doc-101'})
print(f"Cross-tenant delegate: {grant}")

## 16. Expiry
Tokens are strictly time-bound.

In [ ]:
grant_exp = ds.issue(auth_alice, auth_agent, 'document-service', {'read'}, {'doc-101'}, ttl_minutes=5)
clock.advance(6)
decision = ds.verify(grant_exp.grant, 'research-agent', 'document-service', 'acme', 'read', 'doc-101')
print(f"Expired Token: {decision.reason}")

## 17. No Ambient-Authority Fallback
If delegation fails, the service MUST NOT fall back to its ambient privileges. It must fail closed.

In [ ]:
# Reset clock for valid tests
clock.now = datetime(2025, 1, 1, 12, 0, tzinfo=timezone.utc)
res = app.answer_secure(auth_alice, 'doc-secret')
print(f"Secure Fail-Closed Result: {res.terminal_state}")
events = [e for e in audit.events if e.resource_id == 'doc-secret']
print(f"Audit shows DENY: {events[-1].decision} ({events[-1].reason})")

## 18. Separate Service and Delegated Entry Points
Execution mode must come from trusted routing, not request data. The delegated `get_document` method has no `mode` parameter. Background work uses the separate `run_service_read` entry point, which passes the service's infrastructure-authenticated identity to storage.

In [ ]:
try:
    secure_docs.get_document(None, None, 'doc-secret', 'req-mode', mode='service')
except TypeError as exc:
    print(f"Request cannot select service mode: {exc}")

service_result = secure_docs.run_service_read(auth_job, 'doc-101', 'job-1')
print(f"Trusted service entry point: {service_result}")
print(f"Service audit: {audit.events[-1].reason}")

## 19. First Secure Delegated Read
Now let's see a valid delegated read through the Secure API.

In [ ]:
res = app.answer_secure(auth_alice, 'doc-101', 'req-secure-1')
print(f"Valid Secure Read: {res.answer}")

## 20. Multi-Hop Token Exchange
Notice how the Document Service couldn't use the Research Agent's token for Storage? It had to perform a **Token Exchange**. Let's simulate that manually.

In [ ]:
parent = ds.issue(auth_alice, auth_agent, 'document-service', {'read', 'comment'}, {'doc-101'}, ttl_minutes=60)
child = ds.exchange(parent.grant, auth_doc, next_audience='storage-service', requested_operations={'read'}, requested_resources={'doc-101'}, requested_ttl_minutes=10)
print(f"Child Audience: {child.grant.audience}")
print(f"Child Delegate: {child.grant.delegate_id}")

## 21. Parent-Child Grant Chain
The child grant maintains a cryptographic or deterministic link to the parent grant.

In [ ]:
print(f"Parent ID: {parent.grant.grant_id}")
print(f"Child Parent ID: {child.grant.parent_grant_id}")

## 22. Expiry Attenuation
During exchange, a child token cannot outlive its parent. It is clamped to `min(requested_expiry, parent_expiry)`.

In [ ]:
child_long = ds.exchange(parent.grant, auth_doc, 'storage-service', {'read'}, {'doc-101'}, requested_ttl_minutes=120)
print(f"Parent Expiry: {parent.grant.expires_at}")
print(f"Child Expiry : {child_long.grant.expires_at}")

## 23. Scope-Expansion Attack
If a compromised intermediate service tries to ask for more permissions during exchange, it is denied.

In [ ]:
bad_child = ds.exchange(parent.grant, auth_doc, 'storage-service', {'read', 'delete'}, {'doc-101', 'doc-102'}, requested_ttl_minutes=10)
print(f"Scope Expansion Result: {bad_child}")

## 24. Workload Impersonation Attack
If an attacker tries to call the backend by simply passing a string ID without an AuthenticatedWorkload context, it fails.

In [ ]:
res = storage.read_object(caller=None, grant=child.grant, resource_id='doc-101', correlation_id='req-impersonate')
print(f"Impersonation Result: {res}")
print(f"Audit: {audit.events[-1].reason}")

## 25. Audit/Attribution Trace
Let's look at the full audit trace for the successful secure multi-hop request. Notice how identities shift across hops, but Alice is preserved throughout.

In [ ]:
events = [e for e in audit.events if e.correlation_id == 'req-secure-1']
for e in events:
    print(f"Hop {e.delegation_depth}: {e.workload_id} -> {e.audience} | {e.operation} {e.resource_id} | {e.lifecycle_state}")
    print(f"  Principal: {e.principal_id}")
    print(f"  Grant Chain: {e.delegation_id} (Parent: {e.parent_delegation_id})\n")

## 26. Adversarial Matrix
Run `python3 01_identity_propagation.py` in your terminal to execute the full adversarial scenario set.

## 27. Exercises
1. Extend `AuthenticatedServiceJob` with expiry and revocation, then test both exact boundary conditions.
2. Inject a token-exchange failure and verify that the audit trail records a terminal blocked event.
3. Explain why a registry lookup authorizes attributes but cannot authenticate a caller.

## 28. Production Mapping
Real token exchange (RFC 8693) uses authorization-server policy and verifiable credentials. Audience restriction and sender-constrained tokens reduce replay risk. This lab models the control semantics with canonical in-memory contexts; production systems must verify signed or opaque credentials at network boundaries.

References: RFC 8693, RFC 8707, RFC 9700, and the SPIFFE workload identity overview.